<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*



The problem I am trying to solve is deciding which pages out of the 30,000 pages should be looked at first by a FlyRank editor. At the moment, the queue is based on hand-written rules and flags. My approach is to use the model to re-rank that queue so the pages that look more urgent are placed higher.

The unit being scored is one page, using its 90-day trailing data. The dataset covers 32 pseudonymized clients, so the model is not just learning from pages belonging to one client. The output is a ranking score, not a final decision. The top 50 pages would be reviewed by a person, who can then decide whether a page should be refreshed, monitored, expanded, or left alone. The model should not be publishing or changing content by itself.

A healthy page being sent for review still costs an editor time, so I want the top of the queue to contain pages that are actually worth checking. At the same time, missing a real declining page could mean losing traffic without anyone noticing it quickly. That tradeoff is why I used Precision@50 rather than recall as the main metric. I want the pages ranked highest to be useful for the editor's limited review time.

The rule-based approach has a fairly obvious weakness once I looked at individual examples. It uses thresholds, but it does not really weigh signals against each other. Age can say "this page needs attention" while performance says almost the opposite.

For example, page **`cf56e2e2e282`** was more than 181 days old and had 61,678 impressions, so the rule marked it **HIGH_STALE_DECOY** and pushed it to the top of the priority list. But when I looked at the other numbers, the page was ranking **#1** and had a **15% CTR**. Those are strong performance signals. So in this case, the rule is basically saying "fix this" because the page is old and gets a lot of impressions, while ignoring the fact that it is already performing very well. The problem is not that the age signal is useless; it is that the rule has no way to balance age against actual performance.

The model is therefore being used to improve the **ordering of the review queue**, not to explain the cause of a decline. A high score means that a page deserves attention sooner. It does not mean that I know why the page is declining or that I already know what the editor should change.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:

#IMPORTS

import os
import json
import subprocess

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
# Step 1 — Get the Dataset

STARTER_REPO = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
if not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1", STARTER_REPO, "flyrank-ml-internship-starter"], check=True)


df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

In [ ]:
# Step 2 —  Basic dataset checks
print(df.columns)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df.shape, df["client_id"].nunique())
print(df["is_declining_label"].mean())

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')
(30000, 45) 32
0.5420666666666667


## Dataset and release

* **Release:** `content_refresh_anonymized.csv`
* The dataset has **30,000 pages** and **44 original columns**. I also created `is_declining_label` during preprocessing, which is why the final dataframe has 45 columns.
* There are **32 pseudonymized clients** in the data.
* The page-level data uses a **trailing 90-day window**, so the features describe recent performance rather than a single day's result.
* The target is `is_declining_label`. Its mean is **0.542**, meaning about **54.2% of the pages are labelled as declining**.
* I did not use `trend_direction` or `trend_pct` as model features because they are directly related to the label. Including them would give the model information about the answer it is supposed to predict.
* I also left out existing product scoring fields such as `health_score` and other product flags where they were part of the existing baseline. Using those would make the model too close to the current rule instead of testing whether the underlying page features can improve the ranking.
* `client_id` was kept for the grouped evaluation, but not used as a predictive feature. The raw page IDs were also not useful for prediction, so they were excluded.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



I kept the final model fairly simple on purpose. The goal here is not to build the most complicated model possible, but to see whether a small set of useful signals can improve the existing content-refresh queue.

The target is `is_declining_label`, where a page is labelled 1 when its `trend_direction` is `"down"`. I excluded `trend_direction` and `trend_pct` from the model because they are directly related to how the label is defined. I also left out the rule-based product flags such as `stale_flag`, `low_ctr_flag`, `is_decoy`, and `baseline_score`. Those would make the model partly learn the existing baseline instead of giving an independent comparison. The raw IDs are also not used as predictors; `client_id` is only used to keep pages from the same client together during validation.

The final feature set is:

- `freshness_tier_enc` — a simple representation of how old the page is.
- `avg_position` — where the page is ranking.
- `ctr` — how often impressions turn into clicks.
- `impressions_90d` — how much search traffic the page has received.
- `avg_position_missing` — marks pages where `avg_position` is 0. I treated 0 as a missing/unranked value rather than a real ranking position.

I kept the rule-based score as the baseline. It uses the same stale-page, low-CTR, and high-impression conditions from the earlier work, so the model is being compared against the actual rule that it is supposed to improve rather than against a different benchmark.

For validation, I used 5-fold `GroupKFold` with `client_id` as the grouping variable. This means that pages from the same client cannot appear in both the training and test part of a fold. I chose this because a random split could make the result look better if the model sees other pages from the same client during training.

The main metric is Precision@50 because the practical use case is a small editor queue. The model is ranking pages for review, not making an automatic publishing decision, so I care about how many of the first 50 pages are actually declining.

I also checked the feature set for leakage before comparing models. In addition to checking that label-derived and baseline columns were excluded, I ran a deliberate leakage test by putting `trend_pct` back into the features. This is not part of the real model. It is a sanity check to make sure the evaluation reacts when target information is deliberately introduced.

In [ ]:
#Initialize the feature encoding
tier_order = ["0-30", "31-90", "91-180", "181+"]

df["freshness_tier_enc"] = df["freshness_tier"].map(
    {tier: i for i, tier in enumerate(tier_order)}
)

In [ ]:
#Initialize the missing-position feature
df["avg_position_missing"] = (
    df["avg_position"] == 0
).astype(int)

In [ ]:
#Initialize the baseline constants
STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005
IMPRESSION_DECOY_LEVEL = 5000

In [ ]:
#create the baseline flags
df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)

df["low_ctr_flag"] = (
    (df["avg_position"] <= 20)
    & (df["ctr"] < CTR_THRESHOLD)
    & (df["impressions_90d"] >= 500)
)

df["is_decoy"] = (
    (df["freshness_tier"] == "181+")
    & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)
)

In [ ]:
#Initialize the baseline score
def calculate_score(row):
    if row["is_decoy"]:
        return 3
    if row["stale_flag"] and row["low_ctr_flag"]:
        return 2
    if row["stale_flag"] or row["low_ctr_flag"]:
        return 1
    return 0

df["baseline_score"] = df.apply(
    calculate_score,
    axis=1
)

In [ ]:
#Initializing my validation
N_SPLITS = 5

gkf = GroupKFold(
    n_splits=N_SPLITS
)

In [ ]:
#Initialize Precision@50
def precision_at_k(
    sub_df,
    score_col,
    k=50,
    tiebreak_col="impressions_90d"
):
    ranked = sub_df.sort_values(
        [score_col, tiebreak_col],
        ascending=[False, False]
    )

    top_k = ranked.head(k)

    return top_k["is_declining_label"].mean()

In [ ]:
#Initialize your FINAL feature list
FEATURES = [
    "freshness_tier_enc",
    "avg_position",
    "ctr",
    "impressions_90d",
    "avg_position_missing"
]

In [ ]:
print("\nFinal features:")
print(FEATURES)


Final features:
['freshness_tier_enc', 'avg_position', 'ctr', 'impressions_90d', 'avg_position_missing']


In [ ]:
print("\nLabel counts:")
print(df["is_declining_label"].value_counts())


Label counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [ ]:
print("\nBaseline score distribution:")
print(df["baseline_score"].value_counts().sort_index())


Baseline score distribution:
baseline_score
0    19915
1     9603
2      476
3        6
Name: count, dtype: int64


In [ ]:
print("\nGroupKFold check:")

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"]),
    start=1
):
    train_clients = set(df.iloc[train_idx]["client_id"])
    test_clients = set(df.iloc[test_idx]["client_id"])

    print(
        f"Fold {fold}: "
        f"train rows={len(train_idx)}, "
        f"test rows={len(test_idx)}, "
        f"train clients={len(train_clients)}, "
        f"test clients={len(test_clients)}, "
        f"overlap={len(train_clients & test_clients)}"
    )


GroupKFold check:
Fold 1: train rows=22992, test rows=7008, train clients=31, test clients=1, overlap=0
Fold 2: train rows=24269, test rows=5731, train clients=25, test clients=7, overlap=0
Fold 3: train rows=24247, test rows=5753, train clients=24, test clients=8, overlap=0
Fold 4: train rows=24245, test rows=5755, train clients=24, test clients=8, overlap=0
Fold 5: train rows=24247, test rows=5753, train clients=24, test clients=8, overlap=0


### Leakage Check

In [ ]:
def make_rf():
    return RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

In [ ]:
# Leakage check
# The real model must not use trend_direction, trend_pct,
# baseline/product flags, or overlapping time-window features.

# Columns that should never be used as model features
BANNED = [
    "trend_direction",
    "trend_pct"
]

PRODUCT_FLAGS = [
    "stale_flag",
    "low_ctr_flag",
    "is_decoy",
    "baseline_score"
]

WINDOW_TERMS = [
    "last_30d",
    "prev_30d",
    "prev30"
]


In [ ]:

# Check 1 — label-derived columns
banned_found = [
    col for col in FEATURES
    if col in BANNED
]



# Check 2 — baseline/product flags
product_flags_found = [
    col for col in FEATURES
    if col in PRODUCT_FLAGS
]


# Check 3 — overlapping time-window features
window_features_found = [
    col for col in FEATURES
    if any(term in col for term in WINDOW_TERMS)
]



Label-derived columns: CLEAN
Product/baseline flags: CLEAN


In [ ]:

# Deliberate leakage injection test


# Add trend_pct temporarily.
# This should NOT be part of the real model.
df["_leak_probe"] = df["trend_pct"]

LEAKY_FEATURES = FEATURES + ["_leak_probe"]


# Same grouped validation design used for the model
gkf = GroupKFold(n_splits=5)


# Same Random Forest settings used in the capstone
def make_rf():
    return RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )


# Precision@50 helper
def precision_at_k(sub_df, score_col, k=50):
    ranked = sub_df.sort_values(
        [score_col, "impressions_90d"],
        ascending=[False, False]
    )

    top_k = ranked.head(k)

    return top_k["is_declining_label"].mean()


# First calculate the honest feature-set performance
honest_p50s = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    rf = make_rf()

    rf.fit(
        train_fold[FEATURES],
        train_fold["is_declining_label"]
    )

    test_fold["honest_score"] = rf.predict_proba(
        test_fold[FEATURES]
    )[:, 1]

    honest_p50s.append(
        precision_at_k(
            test_fold,
            "honest_score"
        )
    )


# Now repeat the same evaluation with trend_pct deliberately injected
leak_p50s = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"])
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    rf = make_rf()

    rf.fit(
        train_fold[LEAKY_FEATURES],
        train_fold["is_declining_label"]
    )

    test_fold["leak_score"] = rf.predict_proba(
        test_fold[LEAKY_FEATURES]
    )[:, 1]

    leak_p50s.append(
        precision_at_k(
            test_fold,
            "leak_score"
        )
    )




In [ ]:
# Print results
print(
    "Label-derived columns:",
    banned_found if banned_found else "CLEAN"
)
print(
    "Product/baseline flags:",
    product_flags_found if product_flags_found else "CLEAN"
)
print(
    "Overlapping-window features:",
    window_features_found if window_features_found else "CLEAN"
)

print("\nLeakage injection test")
print(
    f"Normal feature set:      "
    f"{np.mean(honest_p50s):.3f} ± {np.std(honest_p50s):.3f}"
)

print(
    f"With trend_pct injected: "
    f"{np.mean(leak_p50s):.3f} ± {np.std(leak_p50s):.3f}"
)


# Remove the artificial leakage column
df.drop(
    columns=["_leak_probe"],
    inplace=True
)

print("\nLeakage probe removed from dataframe.")

Label-derived columns: CLEAN
Product/baseline flags: CLEAN
Overlapping-window features: CLEAN

Leakage injection test
Normal feature set:      0.872 ± 0.069
With trend_pct injected: 1.000 ± 0.000

Leakage probe removed from dataframe.


I also tested whether the evaluation could catch target leakage instead of just assuming that the feature list was clean. I retrained the same Random Forest using the same 5-fold client-grouped split and Precision@50, but added `trend_pct` deliberately. The normal feature set scored **0.872 ± 0.069**, while the leakage version reached **1.000 ± 0.000**. That large jump shows that `trend_pct` gives the model information about the target in disguise, so including it would let the model cheat. I therefore kept it out of the final feature set; this test gives actual evidence for that decision rather than relying only on a column-name check.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
baseline_p50s = []
rf_p50s = []

gkf = GroupKFold(n_splits=5)

In [ ]:


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df, groups=df["client_id"]),
    start=1
):
    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    # Baseline score
    baseline_p50 = precision_at_k(
        test_fold,
        "baseline_score"
    )

    baseline_p50s.append(baseline_p50)

    # Random Forest
    rf = make_rf()

    rf.fit(
        train_fold[FEATURES],
        train_fold["is_declining_label"]
    )

    test_fold["rf_score"] = rf.predict_proba(
        test_fold[FEATURES]
    )[:, 1]

    rf_p50 = precision_at_k(
        test_fold,
        "rf_score"
    )

    rf_p50s.append(rf_p50)

    print(
        f"Fold {fold}: "
        f"Baseline P@50={baseline_p50:.3f}, "
        f"RF P@50={rf_p50:.3f}"
    )

Fold 1: Baseline P@50=0.740, RF P@50=0.940
Fold 2: Baseline P@50=0.960, RF P@50=0.800
Fold 3: Baseline P@50=0.620, RF P@50=0.900
Fold 4: Baseline P@50=0.900, RF P@50=0.940
Fold 5: Baseline P@50=0.720, RF P@50=0.780


In [ ]:
fold_results = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "Baseline P@50": [0.740, 0.960, 0.620, 0.900, 0.720],
    "Random Forest P@50": [0.940, 0.800, 0.900, 0.940, 0.780]
})

display(fold_results)

display(fold_results)

,Fold,Baseline P@50,Random Forest P@50
0,1,0.74,0.94
1,2,0.96,0.80
2,3,0.62,0.90
3,4,0.90,0.94
4,5,0.72,0.78


,Fold,Baseline P@50,Random Forest P@50
0,1,0.74,0.94
1,2,0.96,0.80
2,3,0.62,0.90
3,4,0.90,0.94
4,5,0.72,0.78


In [ ]:
# Overall results
baseline_p50 = np.mean(baseline_p50s)
baseline_p50_std = np.std(baseline_p50s)

model_p50 = np.mean(rf_p50s)
model_p50_std = np.std(rf_p50s)

base_rate = df["is_declining_label"].mean()

In [ ]:

print("\nOverall results")
print(
    f"Baseline P@50: "
    f"{baseline_p50:.3f} ± {baseline_p50_std:.3f}"
)

print(
    f"Random Forest P@50: "
    f"{model_p50:.3f} ± {model_p50_std:.3f}"
)

print(
    f"Base rate: "
    f"{base_rate:.3f}"
)


Overall results
Baseline P@50: 0.788 ± 0.124
Random Forest P@50: 0.872 ± 0.069
Base rate: 0.542


### Results

The Random Forest did better than the existing rule-based baseline overall, although the result was not the same in every fold.

| Method | Precision@50 | What it means |
|---|---:|---|
| Rule-based baseline | **0.788 ± 0.124** | About 79% of the first 50 pages were actually labelled as declining. |
| Random Forest | **0.872 ± 0.069** | About 87% of the first 50 pages were labelled as declining. |
| Base rate | **0.542** | About 54% of all pages in the dataset were labelled as declining. |

The Random Forest improved Precision@50 by about **8.4 percentage points** compared with the baseline. Its smaller standard deviation also suggests that its results were more consistent across the five client-grouped folds.

The fold results were not perfect. The Random Forest was worse than the baseline in Fold 2 (0.800 vs 0.960), but it performed better in the other four folds. This is important because the grouped split shows that the model does not perform equally well for every client group.

Overall, the result supports using the Random Forest as a **ranking model for the editor queue**, but it does not mean that every high-ranked page is actually declining. The model is useful for prioritising which pages should be checked first, rather than making the final decision automatically.

## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

There are a few things I would not claim from these results.

First, this is a **correlation-based ranking model**, not a causal model. The results show which pages tend to have signals associated with the declining label. They do not show that one of these features actually caused a page to decline, and I cannot use this model to claim that it predicts or explains Google's algorithm.

Second, the evaluation uses one dataset and one 90-day data window. I did not test the model across different time periods, so I cannot say yet whether the same results would hold if search behaviour or client performance changed.

Precision@50 also depends on the size of the review queue. I used 50 because that matches the practical idea of giving an editor a small list to work through. The base rate is therefore important for context: **54.2%** of all pages are labelled as declining, compared with **87.2%** in the Random Forest's top 50 pages on average.

There is also variation between client groups. The Random Forest performed better than the baseline overall, but it was not better in every fold. This means the overall average should not be treated as a guarantee for every client.

Finally, the model is meant to help an editor decide what to look at first. It is **not a replacement for editorial judgment**. A high-ranked page still needs to be checked by a person before deciding whether it should be refreshed, monitored, expanded, or left alone.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The model should be used to make the review queue easier to work through, rather than turning its score into an automatic action.

### 1. Review the highest-ranked pages first

**Confidence: High**

Start with the pages at the top of the model's ranking. These pages have the strongest combination of signals used by the model and are the best starting point when editor time is limited.

For rank #1 tomorrow, the FlyRank editor should open the page and check the actual search performance, content quality, intent match, and whether there is an obvious reason it may need attention. The editor then makes the final decision. The model only decides that the page is worth looking at early.

### 2. Check high-priority pages for a real content problem

**Confidence: Medium**

A high model score does not automatically mean that the page needs a full refresh. The editor should check whether the page has a genuine issue, such as outdated information, weak search intent coverage, or poor engagement compared with what would normally be expected.

If there is no clear problem, the page can be moved down or left alone instead of being changed just because the model ranked it highly.

### 3. Keep lower-ranked pages in the queue for later review

**Confidence: Medium**

Pages below the first group should not be treated as completely healthy. They are simply lower priority when compared with the pages above them.

Once the highest-priority pages have been reviewed, editors can work down the ranking and use the same checks. This keeps the model as a prioritisation tool rather than a yes/no decision system.

### What the ranking should and should not do

The ranking should answer:

> **"Which pages should I look at first?"**

It should not answer:

> **"What exactly caused this page to decline?"**

and it should not automatically decide:

> **"This page must be refreshed."**

The final action stays with the FlyRank editor.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The main artifacts are the charts and result tables from the grouped evaluation. I kept the results tied to the same five client-grouped folds used for the final model comparison.

The comparison table shows the overall baseline, Random Forest, and base rate results. The fold table keeps the individual results visible so the overall mean and variation can be checked.

In [ ]:

CHART_DIR = "outputs/charts"

print("Chart folder exists:", os.path.isdir(CHART_DIR))

svg_files = sorted(
    file for file in os.listdir(CHART_DIR)
    if file.lower().endswith(".svg")
)

print("\nSVG files found:")
for file in svg_files:
    print("-", file)

print("\nNumber of SVG files:", len(svg_files))

In [ ]:
comparison_table = pd.DataFrame({
    "Method": [
        "Rule-based baseline",
        "Random Forest",
        "Base rate"
    ],
    "Precision@50": [
        0.788,
        0.872,
        0.542
    ],
    "Std": [
        0.124,
        0.069,
        None
    ]
})

display(comparison_table)

In [ ]:
display(fold_results)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.